# LangGraph

**Domain:** Agentic AI  ·  **runnable:** yes

A refresher on **LangGraph** — a library for building **stateful, multi-step agents and workflows as graphs**. Where [[langchain]]'s LCEL composes *linear* dataflow (`prompt | model | parser`), LangGraph models **nodes, edges, and a shared state object** so you can express **cycles, branches, persistence, and human-in-the-loop** — the control flow real agents need.

## 1. What & Why

**LangGraph** is a low-level orchestration framework (from the LangChain team, but usable standalone) for building agentic applications as a **directed graph of steps over a shared state**. You define a `State` schema, write **nodes** (plain functions that read state and return updates), wire them with **edges** (including *conditional* edges that branch on state), and `compile()` the graph into a runnable.

**The problem it solves: agents are loops, not pipelines.** A real agent calls a model, the model asks for a tool, you run the tool, you feed the result back, the model decides whether it's done or needs another tool — and that loop runs an unknown number of times. You can't express "keep going until the model stops asking for tools" cleanly with `|` pipes. LangChain's old `AgentExecutor` hid this loop in a black box you couldn't inspect or customize. LangGraph makes the loop **explicit, inspectable, and resumable**.

**What you get that a plain LCEL chain doesn't:**
- **Cycles.** Edges can point backward, so an agent can revisit a node until a condition is met.
- **Durable state + checkpointing.** A `checkpointer` snapshots state after every step, keyed by a `thread_id`. Crash, restart, or pause for a day — resume exactly where you left off. This is also how you get conversation memory for free.
- **Human-in-the-loop.** `interrupt()` (or `interrupt_before`/`interrupt_after`) pauses the graph, surfaces state for a human to inspect/edit/approve, then resumes.
- **Streaming of intermediate steps.** Stream state updates *and* tokens, not just the final answer — so you can show the agent's reasoning as it happens.
- **Controllability.** You see and own every node and edge. No hidden agent loop.

**Reach for LangGraph when:**
- You're building a **tool-using agent** with a real call→act→observe loop (the `AgentExecutor` replacement).
- You need **branching/cyclic workflows**: routers, retries, reflection loops, plan-and-execute, multi-agent supervisors.
- You need **persistence** (resume across sessions), **human approval gates**, or to **stream intermediate progress**.

**Skip it when:**
- Your flow is a **straight line** (`retrieve → prompt → model → parse`) — that's [[langchain]]'s LCEL, fewer moving parts.
- You make **one model call**, or a single tool round-trip — the raw provider SDK is simpler.
- You want a **high-level, opinionated** multi-agent framework with roles/tasks baked in — look at [[crewai]] or [[autogen]]; LangGraph is deliberately lower-level.

## 2. Mental Model

Think of LangGraph as **a state machine, or a flowchart you can actually run.**

There's one **shared state object** (a `TypedDict`) that flows through the graph. Each **node** is a function `state -> partial update`; LangGraph **merges** that update back into the state (using a *reducer* per field — replace by default, or *append* for things like message lists). **Edges** decide where to go next; a **conditional edge** is a function that reads the state and returns the name of the next node. Special sentinels `START` and `END` mark entry and exit.

```
                 ┌─────────── shared state (a TypedDict) ───────────┐
                 │  {"messages": [...], "steps": 3, ...}            │
                 └──────────────────────────────────────────────────┘
                                     │  each node returns a PARTIAL update,
                                     ▼  merged in by per-field reducers
   START ──▶ ┌───────┐   should_continue?   ┌────────┐
             │ agent │ ───────────────────▶ │ tools  │ ──┐
             │ (LLM) │ ◀───────────────────────────────┘ │   loop back
             └───────┘        (conditional edge)         │
                 │ "done"                                 │
                 ▼                                        │
                END   ◀───────────────────────────────────┘
```

The agent node calls the model; a **conditional edge** asks "did the model request a tool?" — if yes, route to the `tools` node and **loop back** to the agent with the result; if no, route to `END`. That backward edge — the cycle — is the thing LCEL pipes can't express.

One sentence: **state is a typed dict with per-field merge rules, nodes are functions that return partial updates, edges (some conditional) decide the next node, and `compile()` turns the whole graph into a runnable you can invoke, stream, checkpoint, and interrupt.**

## 3. Key Concepts

| Concept | What it is |
|---|---|
| **`StateGraph(State)`** | The builder. You give it a **state schema** (a `TypedDict`), then add nodes and edges, then `.compile()` it into a runnable graph. |
| **State schema** | A `TypedDict` describing the data that flows through the graph. Each field can carry a **reducer** via `Annotated[type, reducer]` that says how updates merge. |
| **Reducer** | The merge function for a state field. Default = **overwrite**. The common one is `add_messages` (or `operator.add`) to **append** to a list instead of replacing it — essential for accumulating chat history or tool results. |
| **Node** | A function `(state) -> dict`. It reads the current state and returns a **partial** update (only the keys it changes). Nodes are added with `graph.add_node("name", fn)`. |
| **Edge** | A wire between nodes. `add_edge("a", "b")` always goes a→b. `START` / `END` are the entry/exit sentinels (`add_edge(START, "agent")`). |
| **Conditional edge** | `add_conditional_edges("agent", router_fn, {...})`. `router_fn(state)` returns a key (often the next node's name); the mapping turns it into a destination. This is how you **branch and loop**. |
| **`compile(checkpointer=...)`** | Freezes the graph into a runnable exposing `invoke` / `stream` / `astream`. Pass a checkpointer to enable persistence. |
| **Checkpointer** | Persistence backend that snapshots state after each step, keyed by `thread_id` (passed in `config={"configurable": {"thread_id": ...}}`). `MemorySaver` (in-RAM) for dev; `SqliteSaver`/`PostgresSaver` for prod. Gives you resume + memory. |
| **`MessagesState`** | A prebuilt state schema with one field — `messages: Annotated[list, add_messages]`. The default starting point for chat agents so you don't redefine it every time. |
| **`create_react_agent`** | A prebuilt (`langgraph.prebuilt`) that builds the standard agent loop (model ↔ tools) for you in one line — the modern replacement for LangChain's `AgentExecutor`. |
| **`interrupt` / `interrupt_before`** | Human-in-the-loop. Pause the graph (before/after a node, or dynamically) so a human can inspect/edit/approve state, then resume with the same `thread_id`. |
| **`Command`** | A return value from a node that can set state **and** direct control flow (`goto`) in one object — an alternative to separate conditional edges. |

**Mental shortcut:** *nodes do work, the state carries data, edges decide what's next, reducers say how updates merge, the checkpointer makes it durable.*

## 4. Setup

LangGraph is its own package; you add a model provider only when you want to call a real LLM.

```bash
pip install langgraph                       # the graph runtime + prebuilts
pip install langchain-openai                # one provider (or -anthropic, …) for real LLM nodes
pip install langgraph-checkpoint-sqlite     # optional: durable (on-disk) checkpointing
```

To call a model you need that provider's key, e.g. `export OPENAI_API_KEY=...`. LangGraph itself needs **no key** — nodes are just functions, so the graph machinery runs fully offline.

Examples 1 & 2 below are **dependency-free**: they reimplement LangGraph's core (`StateGraph`, reducers, conditional edges, the agent loop) in ~60 lines of plain Python, so the notebook executes anywhere with no install or API key. Example 3 shows the **real LangGraph code** and runs it only if `langgraph` + `langchain-openai` + an `OPENAI_API_KEY` are all present.

In [ ]:
# Installs are optional — examples 1 & 2 are pure stdlib and run offline.
# Uncomment to get the real library used in example 3:
# %pip install langgraph langchain-openai

import importlib.util, os

has_langgraph = importlib.util.find_spec("langgraph") is not None
has_openai    = importlib.util.find_spec("langchain_openai") is not None
has_key       = bool(os.getenv("OPENAI_API_KEY"))
print("langgraph installed:       ", has_langgraph)
print("langchain-openai installed:", has_openai)
print("OPENAI_API_KEY present:    ", has_key)

## 5. Worked Examples

### Example 1 — A `StateGraph` from scratch (state, nodes, reducers, edges)

LangGraph's core is small: a typed **state**, **nodes** that return partial updates, per-field **reducers** that merge those updates, and **edges** that pick the next node. Below we reimplement that core in plain Python — enough to run a real graph — so you can see exactly what `StateGraph(...).compile()` does under the hood. This mirrors `langgraph.graph`.

In [ ]:
from __future__ import annotations
from typing import Any, Callable

START, END = "__start__", "__end__"


class StateGraph:
    """Minimal stand-in for langgraph.graph.StateGraph."""

    def __init__(self, reducers: dict[str, Callable[[Any, Any], Any]] | None = None):
        # reducers[field] = how to merge an update into existing state.
        # A field with no reducer is OVERWRITTEN (LangGraph's default).
        self.reducers = reducers or {}
        self.nodes: dict[str, Callable[[dict], dict]] = {}
        self.edges: dict[str, str] = {}                 # static a -> b
        self.cond: dict[str, Callable[[dict], str]] = {}  # node -> router fn

    def add_node(self, name, fn):  self.nodes[name] = fn; return self
    def add_edge(self, a, b):      self.edges[a] = b; return self
    def add_conditional_edges(self, src, router): self.cond[src] = router; return self
    def set_entry_point(self, name): self.edges[START] = name; return self

    def _merge(self, state: dict, update: dict) -> dict:
        for key, val in update.items():
            if key in self.reducers and key in state:
                state[key] = self.reducers[key](state[key], val)   # e.g. append
            else:
                state[key] = val                                   # overwrite
        return state

    def compile(self):  # returns a runnable
        return CompiledGraph(self)


class CompiledGraph:
    def __init__(self, g: StateGraph):
        self.g = g

    def invoke(self, state: dict, *, max_steps: int = 25) -> dict:
        state = dict(state)
        node = self.g.edges[START]
        for _ in range(max_steps):
            if node == END:
                break
            update = self.g.nodes[node](state)          # node: state -> partial update
            state = self.g._merge(state, update)
            # pick next node: conditional edge wins, else static edge
            node = self.g.cond[node](state) if node in self.g.cond else self.g.edges[node]
        return state


# --- Build a tiny pipeline: two nodes that each append to a list ---------------
import operator

graph = StateGraph(reducers={"log": operator.add})       # "log" field APPENDS
graph.add_node("greet",   lambda s: {"log": [f"hello {s['name']}"], "count": 1})
graph.add_node("shout",   lambda s: {"log": ["GOODBYE"], "count": s["count"] + 1})
graph.set_entry_point("greet")
graph.add_edge("greet", "shout")
graph.add_edge("shout", END)

app = graph.compile()
result = app.invoke({"name": "ada", "log": [], "count": 0})
print("final state:", result)
print("log (appended via reducer):", result["log"])
print("count (overwritten each node):", result["count"])

### Example 2 — A cyclic agent loop with a conditional edge

This is the pattern LCEL pipes *cannot* express: the **agent ↔ tools cycle**. The `agent` node "thinks", a **conditional edge** asks *should we keep going?* — if the agent still wants a tool, route to `tools` and **loop back to `agent`**; otherwise route to `END`. We use a deterministic fake "model" (no API key) so the loop runs end to end with real output. Swap the fake model for `ChatOpenAI` and this is essentially what `create_react_agent` builds.

In [ ]:
# A toy task: the agent must "look up" three facts via a tool, one per turn,
# then finish. The model decides each turn whether another tool call is needed.

FACTS = {"capital": "Paris", "population": "67M", "currency": "EUR"}
WANTED = list(FACTS)

def agent_node(state: dict) -> dict:
    """The 'LLM': decide the next action from what we've gathered so far."""
    gathered = state["facts"]
    missing = [k for k in WANTED if k not in gathered]
    if missing:
        # "request a tool call" for the next missing fact
        return {"next_tool": missing[0],
                "scratch": [f"agent: I still need '{missing[0]}', calling tool"]}
    return {"next_tool": None,
            "scratch": [f"agent: have all facts {gathered}, answering"]}

def tools_node(state: dict) -> dict:
    """Execute the requested tool and write its result into state."""
    key = state["next_tool"]
    return {"facts": {key: FACTS[key]},
            "scratch": [f"tool:  looked up {key} -> {FACTS[key]}"]}

def should_continue(state: dict) -> str:
    """Conditional edge: loop to tools, or stop."""
    return "tools" if state["next_tool"] is not None else END

def merge_dicts(old: dict, new: dict) -> dict:
    return {**old, **new}

agent = StateGraph(reducers={"facts": merge_dicts, "scratch": operator.add})
agent.add_node("agent", agent_node)
agent.add_node("tools", tools_node)
agent.set_entry_point("agent")
agent.add_conditional_edges("agent", should_continue)   # agent -> tools | END
agent.add_edge("tools", "agent")                         # tools -> agent  (the CYCLE)

app = agent.compile()
final = app.invoke({"facts": {}, "next_tool": None, "scratch": []})

print("trace:")
for line in final["scratch"]:
    print(" ", line)
print("\ncollected facts:", final["facts"])

### Example 3 — The real thing: `create_react_agent` (or a hand-built `StateGraph`)

This is the production pattern with real classes. It calls a model, so it runs **only** if `langgraph` + `langchain-openai` are installed *and* `OPENAI_API_KEY` is set; otherwise it prints the exact code you'd write. Note the shape is identical to Example 2 — an agent↔tools cycle over a shared `messages` state — just built for you by `create_react_agent`, and the compiled graph still exposes `invoke` / `stream`.

In [ ]:
SNIPPET = """
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    \"\"\"Return the weather for a city.\"\"\"
    return f"It's sunny in {city}, 22C."

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# One line builds the agent<->tools graph (the loop from Example 2),
# with a checkpointer so the thread_id remembers the conversation.
agent = create_react_agent(model, tools=[get_weather], checkpointer=MemorySaver())

config = {"configurable": {"thread_id": "demo-1"}}
out = agent.invoke({"messages": [("user", "What's the weather in Paris?")]}, config)
print(out["messages"][-1].content)

# Stream intermediate steps instead of waiting for the final answer:
for step in agent.stream({"messages": [("user", "And in Tokyo?")]}, config):
    print(step)
"""

if has_langgraph and has_openai and os.getenv("OPENAI_API_KEY"):
    from langgraph.prebuilt import create_react_agent
    from langchain_openai import ChatOpenAI
    from langchain_core.tools import tool

    @tool
    def get_weather(city: str) -> str:
        """Return the weather for a city."""
        return f"It's sunny in {city}, 22C."

    model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    agent = create_react_agent(model, tools=[get_weather])
    out = agent.invoke({"messages": [("user", "What's the weather in Paris?")]})
    print("Agent:", out["messages"][-1].content)
else:
    print("langgraph + langchain-openai + OPENAI_API_KEY not all present — here is the real code:")
    print(SNIPPET)

## 6. Gotchas & Pitfalls

- **Forgetting reducers = clobbered state.** By default a field is **overwritten** by each node's update. If you want to *accumulate* (messages, tool results, a log), you **must** annotate the field with a reducer — `Annotated[list, add_messages]` or `Annotated[list, operator.add]`. The #1 bug: messages vanish because the field had no reducer and each node replaced the whole list.
- **Nodes return *partial* updates, not the whole state.** Return only the keys you changed (`return {"messages": [reply]}`). Returning the full state — or mutating the input dict in place — fights the reducer machinery. Treat state as immutable; emit a delta.
- **Cycles need a termination condition.** A backward edge with no exit is an infinite loop. Always have a conditional edge that can reach `END`, and consider a `recursion_limit` (LangGraph raises `GraphRecursionError` at the default of 25 steps) as a backstop.
- **`thread_id` is required to use a checkpointer.** Persistence is keyed by `config={"configurable": {"thread_id": ...}}`. Forget it and you get an error or a fresh state every call. Reusing the *same* `thread_id` is exactly how you continue a conversation; a *new* one starts fresh.
- **`MemorySaver` is in-RAM only.** Great for notebooks/tests, but state vanishes when the process exits. For real persistence use `SqliteSaver` / `PostgresSaver` (separate `langgraph-checkpoint-*` packages).
- **`create_react_agent` is convenient but opaque.** It hides the graph. The moment you need custom routing, extra nodes (validation, reflection), or a non-standard loop, drop to a hand-built `StateGraph` — that's the whole point of LangGraph.
- **Conditional-edge router must return a *known* key.** The router function returns a string that has to match a node name (or a key in the mapping dict / `END`). A typo silently routes nowhere — or raises. Keep the destinations small and explicit.
- **Streaming modes differ.** `stream_mode="values"` yields the full state each step, `"updates"` yields only the delta from each node, `"messages"` streams LLM tokens. Picking the wrong mode is why "streaming doesn't show what I expected."
- **It's lower-level than it looks.** LangGraph gives you primitives, not an opinionated agent. Roles, task delegation, and multi-agent choreography are *yours to build* (or reach for [[crewai]]/[[autogen]]). The trade for that control is more wiring.

## 7. When to Use vs Alternatives

| Option | Best for | Trade-offs vs LangGraph |
|---|---|---|
| **LangGraph** | Stateful, **cyclic** agent loops; custom routing/branching; persistence + resume; human-in-the-loop; streaming intermediate steps | Lower-level — you wire nodes/edges/state yourself; overkill for linear flows or one-shot calls |
| **[[langchain]] (LCEL)** | **Linear** dataflow: `retrieve → prompt → model → parse`; RAG chains; provider-agnostic composition | Can't express cycles/branching-with-state; agents (`AgentExecutor`) are legacy → it now *points you to* LangGraph. Often used **together** |
| **[[crewai]]** | High-level **role/task** multi-agent crews; fast to stand up; opinionated | Less control over the exact control flow/state; harder to customize the loop than raw LangGraph |
| **[[autogen]]** | **Conversational** multi-agent systems (agents chatting to solve a task); code-exec focus | Different (conversation-centric) model; LangGraph is graph/state-centric and more deterministic |
| **[[openai-agents-sdk]]** | Lightweight agent loop tied to OpenAI's stack (handoffs, guardrails) | Less provider-agnostic; smaller scope than LangGraph's general graph runtime |
| **[[react]] (raw pattern)** | Understanding/implementing the reason-act loop by hand; tiny apps | No persistence, streaming, HIL, or graph tooling — you rebuild what LangGraph gives you |
| **Raw provider SDK** | One model call or a single tool round-trip | You hand-build the loop, state, retries, and persistence — exactly LangGraph's job |

**Rule of thumb:** **linear flow → [[langchain]] LCEL.** **A loop with state, branching, persistence, or approval gates → LangGraph.** **Want roles/tasks out of the box → [[crewai]]/[[autogen]].** A very common production stack is **LangChain for the components + LangGraph for the control flow + LangSmith for tracing**.

Related notebooks: [[langchain]], [[crewai]], [[autogen]], [[react]], [[openai-agents-sdk]], [[smolagents]], [[llamaindex-agents]].

## 8. Resources

- **Official docs** — https://langchain-ai.github.io/langgraph/
- **Quickstart (build a basic agent)** — https://langchain-ai.github.io/langgraph/agents/agents/
- **Core concepts (low-level graph API)** — https://langchain-ai.github.io/langgraph/concepts/low_level/
- **Persistence & checkpointing** — https://langchain-ai.github.io/langgraph/concepts/persistence/
- **Human-in-the-loop** — https://langchain-ai.github.io/langgraph/concepts/human_in_the_loop/
- **`create_react_agent` / prebuilts** — https://langchain-ai.github.io/langgraph/reference/prebuilt/
- **GitHub** — https://github.com/langchain-ai/langgraph

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
START, END = "__start__", "__end__"


class GraphRecursionError(RuntimeError):
    pass


class StateGraph:
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE